# Content-Based Filtering (Movies)

This notebook builds a content-based recommender for movies with:
- Overview embeddings using `all-mpnet-base-v2`
- Genre similarity via Jaccard
- Keyword similarity via Jaccard
- Actor overlap (weight-reduced for Animation)
- Director direct match
- Collection/franchise direct match
- Bayesian weighted rating prior
- Recency booster from `release_date`
- Adaptive query-aware weights
- Quality-aware structured reranking for better recommendation balance

## 1) Imports

In [2]:
from __future__ import annotations

import ast
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
#from sentence_transformers import SentenceTransformer

## 2) Load Data

In [3]:
DATA_PATH = Path("../../data/processed/movie_metadata_final.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Could not find dataset at {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
print(f"Loaded rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
df.head(3)

Loaded rows: 31,167
Columns: ['title', 'genres', 'id', 'original_title', 'overview', 'popularity', 'original_language', 'production_companies', 'release_date', 'runtime', 'vote_average', 'vote_count', 'collection_name', 'keywords', 'top_5_actors', 'director', 'poster_path', 'has_franchise']


,title,genres,id,original_title,overview,popularity,original_language,production_companies,release_date,runtime,vote_average,vote_count,collection_name,keywords,top_5_actors,director,poster_path,has_franchise
0,Toy Story,"['Animation', 'Comedy', 'Family']",862,Toy Story,"Led by Woody, Andy's toys live happily in his ...",21.946943,en,['Pixar Animation Studios'],1995-10-30,81.0,7.7,5415.0,Toy Story Collection,"['jealousy', 'toy', 'boy', 'friendship', 'frie...","['Tom Hanks', 'Tim Allen', 'Don Rickles', 'Jim...",['John Lasseter'],/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg,1
1,Jumanji,"['Adventure', 'Fantasy', 'Family']",8844,Jumanji,When siblings Judy and Peter discover an encha...,17.015539,en,"['TriStar Pictures', 'Teitler Film', 'Intersco...",1995-12-15,104.0,6.9,2413.0,NaN,"['board game', 'disappearance', ""based on chil...","['Robin Williams', 'Jonathan Hyde', 'Kirsten D...",['Joe Johnston'],/vgpXmVaVyUL7GGiDeiK1mKEKzcX.jpg,0
2,Grumpier Old Men,"['Romance', 'Comedy']",15602,Grumpier Old Men,A family wedding reignites the ancient feud be...,11.712900,en,"['Warner Bros.', 'Lancaster Gate']",1995-12-22,101.0,6.5,92.0,Grumpy Old Men Collection,"['fishing', 'best friend', 'duringcreditssting...","['Walter Matthau', 'Jack Lemmon', 'Ann-Margret...",['Howard Deutch'],/1FSXpj5e8l4KH6nVFO5SPUeraOt.jpg,1


## 3) Preparation

In [4]:
def parse_list_field(value) -> list[str]:
    """Parse stringified list fields like genres, keywords, actors, director."""
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return [str(item).strip().lower() for item in parsed if str(item).strip()]
        except (ValueError, SyntaxError):
            pass
    return [part.strip().lower() for part in text.split(",") if part.strip()]


def parse_set_field(value) -> set[str]:
    return set(parse_list_field(value))


work_df = df.copy()

# Text fields
work_df["overview"] = work_df["overview"].fillna("").astype(str)
work_df["title"] = work_df["title"].fillna("").astype(str)
work_df["collection_name"] = work_df["collection_name"].fillna("").astype(str).str.strip().str.lower()

# Numeric/date fields
work_df["vote_average"] = pd.to_numeric(work_df["vote_average"], errors="coerce")
work_df["vote_count"] = pd.to_numeric(work_df["vote_count"], errors="coerce").fillna(0.0)
work_df["release_date"] = pd.to_datetime(work_df["release_date"], errors="coerce")

# Parsed set/list fields
work_df["_genre_set"] = work_df["genres"].apply(parse_set_field)
work_df["_keyword_set"] = work_df["keywords"].apply(parse_set_field)
work_df["_actor_list"] = work_df["top_5_actors"].apply(parse_list_field)
work_df["_actor_set"] = work_df["_actor_list"].apply(set)
work_df["_director_list"] = work_df["director"].apply(parse_list_field)
work_df["_director_str"] = work_df["_director_list"].apply(lambda d: d[0] if d else "")
work_df["_is_animation"] = work_df["_genre_set"].apply(lambda g: "animation" in g)

# Filter out rows without a title
work_df = work_df[work_df["title"].str.strip() != ""].reset_index(drop=True)

print(f"Usable rows: {len(work_df):,}")
print(f"Rows with rating: {work_df['vote_average'].notna().sum():,}")
print(f"Rows with release date: {work_df['release_date'].notna().sum():,}")
print(f"Animation films: {work_df['_is_animation'].sum():,}")

Usable rows: 31,167
Rows with rating: 31,167
Rows with release date: 31,157
Animation films: 1,431


## 4) Build & Store Overview Embeddings (`all-mpnet-base-v2`)

In [ ]:
MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
model = SentenceTransformer(MODEL_NAME)

overview_texts = work_df["overview"].tolist()
embeddings = model.encode(
    overview_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
).astype(np.float32)

print("Embedding matrix shape:", embeddings.shape)
print("Embedding dtype:", embeddings.dtype)

## 5) Similarity Components, Bayesian Rating & Recency

In [ ]:
def jaccard_similarity(a: set[str], b: set[str]) -> float:
    if not a and not b:
        return 0.0
    union = a | b
    if not union:
        return 0.0
    return len(a & b) / len(union)


def actor_overlap(a: set[str], b: set[str]) -> float:
    """Fraction of shared actors relative to the smaller cast."""
    if not a or not b:
        return 0.0
    shared = len(a & b)
    return shared / min(len(a), len(b))


def bayesian_weighted_rating(R: float, v: float, C: float, m: float) -> float:
    denominator = v + m
    if denominator <= 0:
        return C
    return (v / denominator) * R + (m / denominator) * C


RECENCY_DECAY_YEARS = 12.0
CURRENT_TIMESTAMP = pd.Timestamp.utcnow().tz_localize(None)

# Precompute arrays
genre_sets = work_df["_genre_set"].tolist()
keyword_sets = work_df["_keyword_set"].tolist()
actor_sets = work_df["_actor_set"].tolist()
directors = work_df["_director_str"].to_numpy()
collections = work_df["collection_name"].to_numpy()
is_animation = work_df["_is_animation"].to_numpy()

ratings = work_df["vote_average"].fillna(work_df["vote_average"].mean()).astype(float).to_numpy()
votes = work_df["vote_count"].fillna(0.0).astype(float).to_numpy()

C = float(np.nanmean(ratings))
m = float(np.percentile(votes, 75))

bayesian_scores = np.array(
    [bayesian_weighted_rating(R=ratings[i], v=votes[i], C=C, m=m) for i in range(len(work_df))]
)

bayes_min = float(np.min(bayesian_scores))
bayes_max = float(np.max(bayesian_scores))
if bayes_max > bayes_min:
    bayesian_scores_norm = (bayesian_scores - bayes_min) / (bayes_max - bayes_min)
else:
    bayesian_scores_norm = np.zeros_like(bayesian_scores)

release_dates = work_df["release_date"]
age_years = ((CURRENT_TIMESTAMP - release_dates).dt.days.fillna(365.25 * 100) / 365.25).clip(lower=0.0)
recency_scores = np.exp(-age_years / RECENCY_DECAY_YEARS).to_numpy(dtype=np.float32)

print(f"Global average rating (C): {C:.4f}")
print(f"Minimum votes threshold (m): {m:.2f}")
print(f"Bayesian score range: [{bayes_min:.4f}, {bayes_max:.4f}]")
print(f"Recency decay years: {RECENCY_DECAY_YEARS:.1f}")
print(f"Recency score range: [{recency_scores.min():.4f}, {recency_scores.max():.4f}]")

## 6) Adaptive Weights, Recency Boost & Structured Reranking

In [ ]:
BASE_FEATURE_WEIGHTS = {
    "embedding": 0.33,
    "genre": 0.15,
    "keyword": 0.15,
    "actor": 0.10,
    "director": 0.05,
    "collection": 0.03,
    "bayesian": 0.15,
    "recency": 0.04,
}

RETRIEVAL_K_DEFAULT = 150
MIN_BAYES_SCORE_GATE = 0.55
MIN_BAYES_WITH_STRONG_MATCH = 0.45
MIN_STRONG_METADATA_GATE = 0.70
TITLE_FAMILY_MATCH_THRESHOLD = 0.50
TITLE_STOPWORDS = {
    "the", "a", "an", "of", "and", "or", "to", "in", "on", "for", "with",
    "movie", "film", "part", "chapter", "episode", "story", "stories", "adventure",
    "adventures", "return", "returns", "rise", "rises", "fall", "falls", "vs", "v",
    "ii", "iii", "iv", "v", "vi", "vii", "viii", "ix", "x",
}


def normalize_weights(weights: dict[str, float]) -> dict[str, float]:
    total = float(sum(weights.values()))
    if total <= 0:
        raise ValueError("Weight sum must be > 0")
    return {key: value / total for key, value in weights.items()}


def shift_weight(
    weights: dict[str, float],
    from_key: str,
    to_keys: list[str],
    amount: float,
) -> None:
    if amount <= 0 or not to_keys:
        return
    shift = min(float(amount), float(weights.get(from_key, 0.0)))
    if shift <= 0:
        return
    weights[from_key] -= shift
    per_target = shift / len(to_keys)
    for key in to_keys:
        weights[key] = weights.get(key, 0.0) + per_target


def tokenize_title(title: str) -> set[str]:
    normalized = re.sub(r"[^a-z0-9]+", " ", str(title).lower()).strip()
    tokens = {
        token
        for token in normalized.split()
        if len(token) >= 3 and not token.isdigit() and token not in TITLE_STOPWORDS
    }
    return tokens


def title_family_similarity(a: set[str], b: set[str]) -> float:
    if not a or not b:
        return 0.0
    return len(a & b) / min(len(a), len(b))


def get_feature_weights(q_idx: int) -> dict[str, float]:
    """Query-aware weighting to avoid one signal dominating every title."""
    weights = BASE_FEATURE_WEIGHTS.copy()

    q_genres = genre_sets[q_idx]
    q_keywords = keyword_sets[q_idx]
    q_actors = actor_sets[q_idx]
    q_director = directors_arr[q_idx]
    q_collection = collections_arr[q_idx]
    q_is_animation = bool(is_animation[q_idx])
    q_release_date = work_df.iloc[q_idx]["release_date"]
    overview_word_count = len(str(work_df.iloc[q_idx]["overview"]).split())

    if q_is_animation:
        shift_weight(weights, "actor", ["genre", "keyword"], 0.06)

    if not q_director:
        shift_weight(weights, "director", ["embedding", "genre"], weights["director"])

    if not q_collection:
        shift_weight(weights, "collection", ["embedding", "genre"], weights["collection"])
    else:
        shift_weight(weights, "collection", ["genre", "keyword"], 0.02)

    if len(q_keywords) == 0:
        shift_weight(weights, "keyword", ["embedding", "genre"], weights["keyword"])
    elif len(q_keywords) < 3:
        shift_weight(weights, "keyword", ["embedding", "genre"], 0.05)
    elif len(q_keywords) >= 8:
        shift_weight(weights, "embedding", ["keyword"], 0.05)

    if len(q_actors) <= 1:
        shift_weight(weights, "actor", ["embedding", "genre"], weights["actor"] * 0.6)
    elif len(q_actors) >= 4 and not q_is_animation:
        shift_weight(weights, "embedding", ["actor"], 0.04)

    if len(q_genres) >= 3:
        shift_weight(weights, "embedding", ["genre"], 0.04)

    if overview_word_count < 25:
        shift_weight(weights, "embedding", ["genre", "keyword"], 0.07)

    if pd.isna(q_release_date):
        shift_weight(weights, "recency", ["embedding", "genre"], weights["recency"])

    return normalize_weights(weights)


titles_arr = work_df["title"].to_numpy()
directors_arr = np.asarray(directors, dtype=object)
collections_arr = np.asarray(collections, dtype=object)
bayesian_norm_arr = np.asarray(bayesian_scores_norm, dtype=np.float32)
recency_scores_arr = np.asarray(recency_scores, dtype=np.float32)
title_token_sets = [tokenize_title(title) for title in titles_arr]


def find_query_index(name: str) -> int:
    name_norm = name.strip().lower()
    titles = work_df["title"].str.lower()

    exact = titles[titles == name_norm]
    if len(exact) > 0:
        return int(exact.index[0])

    contains = titles[titles.str.contains(name_norm, regex=False)]
    if len(contains) > 0:
        return int(contains.index[0])

    raise ValueError(f"Could not find movie with name: {name}")


def compute_quality_adjustment(
    embedding_scores: np.ndarray,
    genre_scores: np.ndarray,
    keyword_scores: np.ndarray,
    actor_scores: np.ndarray,
    director_scores: np.ndarray,
    collection_scores: np.ndarray,
    bayes_scores: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    strong_metadata_match = np.maximum.reduce(
        [genre_scores, keyword_scores, actor_scores, director_scores, collection_scores]
    )

    quality_multiplier = 0.45 + 0.55 * bayes_scores

    low_quality_penalty = np.ones_like(bayes_scores, dtype=np.float32)
    low_quality_penalty = np.where(
        (bayes_scores < 0.55) & (strong_metadata_match < 0.55),
        0.75,
        low_quality_penalty,
    )
    low_quality_penalty = np.where(
        (bayes_scores < 0.45) & (strong_metadata_match < 0.50),
        0.60,
        low_quality_penalty,
    )
    low_quality_penalty = np.where(
        (bayes_scores < 0.35) & (strong_metadata_match < 0.35) & (embedding_scores > 0.60),
        0.45,
        low_quality_penalty,
    )

    rescue_bonus = np.ones_like(bayes_scores, dtype=np.float32)
    rescue_bonus = np.where(
        (bayes_scores >= 0.40) & (director_scores > 0),
        1.04,
        rescue_bonus,
    )
    rescue_bonus = np.where(
        (bayes_scores >= 0.45) & (genre_scores > 0.60) & (keyword_scores > 0.30),
        1.03,
        rescue_bonus,
    )

    return quality_multiplier * low_quality_penalty * rescue_bonus, strong_metadata_match


def collection_cap_for_rank(rank_position: int) -> int:
    if rank_position < 10:
        return 2
    if rank_position < 20:
        return 3
    return 4


def director_cap_for_rank(rank_position: int) -> int:
    if rank_position < 10:
        return 2
    if rank_position < 20:
        return 3
    return 4


def title_family_cap_for_rank(rank_position: int) -> int:
    if rank_position < 10:
        return 2
    if rank_position < 20:
        return 3
    return 4


def structured_rerank(candidates_df: pd.DataFrame, q_idx: int, top_k: int) -> pd.DataFrame:
    if candidates_df.empty:
        return candidates_df

    pool = candidates_df.copy().reset_index(drop=True)
    selected_rows: list[pd.Series] = []
    selected_indices: list[int] = []
    collection_counts: dict[str, int] = {}
    director_counts: dict[str, int] = {}
    q_title_tokens = title_token_sets[q_idx]

    while len(selected_rows) < min(top_k, len(pool)):
        rank_position = len(selected_rows)
        best_row_pos = None
        best_rerank_score = -np.inf

        for enforce_caps in (True, False):
            for row_pos in range(len(pool)):
                row = pool.iloc[row_pos]
                candidate_index = int(row["index"])
                candidate_collection = collections_arr[candidate_index]
                candidate_director = directors_arr[candidate_index]
                candidate_title_tokens = title_token_sets[candidate_index]

                same_title_family_count = sum(
                    title_family_similarity(candidate_title_tokens, title_token_sets[idx]) >= TITLE_FAMILY_MATCH_THRESHOLD
                    for idx in selected_indices
                )

                if enforce_caps:
                    if candidate_collection and collection_counts.get(candidate_collection, 0) >= collection_cap_for_rank(rank_position):
                        continue
                    if candidate_director and director_counts.get(candidate_director, 0) >= director_cap_for_rank(rank_position):
                        continue
                    if same_title_family_count >= title_family_cap_for_rank(rank_position):
                        continue

                query_title_family_score = title_family_similarity(q_title_tokens, candidate_title_tokens)
                collection_penalty = 0.12 * collection_counts.get(candidate_collection, 0) if candidate_collection else 0.0
                director_penalty = 0.05 * director_counts.get(candidate_director, 0) if candidate_director else 0.0
                title_family_penalty = 0.12 * same_title_family_count
                if query_title_family_score >= TITLE_FAMILY_MATCH_THRESHOLD:
                    title_family_penalty += 0.08

                if selected_indices:
                    max_genre_redundancy = max(
                        jaccard_similarity(genre_sets[candidate_index], genre_sets[idx])
                        for idx in selected_indices
                    )
                    max_keyword_redundancy = max(
                        jaccard_similarity(keyword_sets[candidate_index], keyword_sets[idx])
                        for idx in selected_indices
                    )
                    max_embedding_similarity = float(np.max(embeddings[selected_indices] @ embeddings[candidate_index]))
                else:
                    max_genre_redundancy = 0.0
                    max_keyword_redundancy = 0.0
                    max_embedding_similarity = 0.0

                extra_low_quality_penalty = 0.0
                if row["bayesian_rating_norm"] < 0.55 and row["strong_metadata_match"] < 0.55:
                    extra_low_quality_penalty = 0.07

                rerank_score = (
                    float(row["quality_adjusted_score"])
                    + 0.02 * float(row["recency_score"])
                    - collection_penalty
                    - director_penalty
                    - title_family_penalty
                    - 0.08 * max_genre_redundancy
                    - 0.10 * max_keyword_redundancy
                    - 0.08 * max_embedding_similarity
                    - extra_low_quality_penalty
                )

                if rerank_score > best_rerank_score:
                    best_rerank_score = rerank_score
                    best_row_pos = row_pos

            if best_row_pos is not None:
                break

        chosen = pool.iloc[best_row_pos].copy()
        chosen["rerank_score"] = best_rerank_score
        selected_rows.append(chosen)

        chosen_index = int(chosen["index"])
        selected_indices.append(chosen_index)

        chosen_collection = collections_arr[chosen_index]
        chosen_director = directors_arr[chosen_index]
        if chosen_collection:
            collection_counts[chosen_collection] = collection_counts.get(chosen_collection, 0) + 1
        if chosen_director:
            director_counts[chosen_director] = director_counts.get(chosen_director, 0) + 1

        pool = pool.drop(index=best_row_pos).reset_index(drop=True)

    return pd.DataFrame(selected_rows).reset_index(drop=True)


def recommend_by_index(
    q_idx: int,
    top_k: int = 10,
    retrieval_k: int = RETRIEVAL_K_DEFAULT,
) -> pd.DataFrame:
    n_total = len(work_df)
    if q_idx < 0 or q_idx >= n_total:
        raise ValueError(f"Invalid q_idx={q_idx}. Must be in [0, {n_total - 1}]")
    if n_total <= 1:
        return pd.DataFrame(columns=["index", "title", "final_score"])

    n_candidates = min(max(retrieval_k, top_k * 6), n_total - 1)

    query_vec = embeddings[q_idx]
    sims = embeddings @ query_vec
    sims[q_idx] = -1.0

    top_indices = np.argpartition(sims, -n_candidates)[-n_candidates:]
    top_indices = top_indices[np.argsort(sims[top_indices])[::-1]]
    ci = top_indices.astype(int)

    q_genres = genre_sets[q_idx]
    q_keywords = keyword_sets[q_idx]
    q_actors = actor_sets[q_idx]
    q_director = directors_arr[q_idx]
    q_collection = collections_arr[q_idx]
    q_title_tokens = title_token_sets[q_idx]

    weights = get_feature_weights(q_idx)

    embedding_scores = sims[ci].astype(np.float32)
    genre_scores = np.array([jaccard_similarity(q_genres, genre_sets[i]) for i in ci], dtype=np.float32)
    keyword_scores = np.array([jaccard_similarity(q_keywords, keyword_sets[i]) for i in ci], dtype=np.float32)
    actor_scores = np.array([actor_overlap(q_actors, actor_sets[i]) for i in ci], dtype=np.float32)
    director_scores = (directors_arr[ci] == q_director).astype(np.float32) if q_director else np.zeros(len(ci), dtype=np.float32)
    collection_scores = (collections_arr[ci] == q_collection).astype(np.float32) if q_collection else np.zeros(len(ci), dtype=np.float32)
    title_family_scores = np.array([title_family_similarity(q_title_tokens, title_token_sets[i]) for i in ci], dtype=np.float32)
    bayes_scores = bayesian_norm_arr[ci]
    recency_candidate_scores = recency_scores_arr[ci]

    base_scores = (
        weights["embedding"] * embedding_scores
        + weights["genre"] * genre_scores
        + weights["keyword"] * keyword_scores
        + weights["actor"] * actor_scores
        + weights["director"] * director_scores
        + weights["collection"] * collection_scores
        + weights["bayesian"] * bayes_scores
        + weights["recency"] * recency_candidate_scores
    )

    quality_adjustment, strong_metadata_match = compute_quality_adjustment(
        embedding_scores=embedding_scores,
        genre_scores=genre_scores,
        keyword_scores=keyword_scores,
        actor_scores=actor_scores,
        director_scores=director_scores,
        collection_scores=collection_scores,
        bayes_scores=bayes_scores,
    )
    quality_adjusted_scores = base_scores * quality_adjustment

    base_ranked = pd.DataFrame({
        "index": ci,
        "title": titles_arr[ci],
        "final_score": base_scores,
        "quality_adjusted_score": quality_adjusted_scores,
        "embedding_score": embedding_scores,
        "genre_score": genre_scores,
        "keyword_score": keyword_scores,
        "actor_score": actor_scores,
        "director_match": director_scores.astype(int),
        "collection_match": collection_scores.astype(int),
        "title_family_score": title_family_scores,
        "bayesian_rating_norm": bayes_scores,
        "recency_score": recency_candidate_scores,
        "strong_metadata_match": strong_metadata_match,
    }).sort_values("quality_adjusted_score", ascending=False).reset_index(drop=True)

    filtered_candidates = base_ranked[
        (base_ranked["bayesian_rating_norm"] >= MIN_BAYES_SCORE_GATE)
        | (
            (base_ranked["bayesian_rating_norm"] >= MIN_BAYES_WITH_STRONG_MATCH)
            & (base_ranked["strong_metadata_match"] >= MIN_STRONG_METADATA_GATE)
        )
    ].reset_index(drop=True)

    if len(filtered_candidates) < top_k:
        filtered_candidates = base_ranked.head(max(top_k * 3, top_k)).copy()

    return structured_rerank(filtered_candidates, q_idx=q_idx, top_k=top_k)


def recommend_by_name(
    name: str,
    top_k: int = 10,
    retrieval_k: int = RETRIEVAL_K_DEFAULT,
) -> pd.DataFrame:
    q_idx = find_query_index(name)
    return recommend_by_index(q_idx, top_k=top_k, retrieval_k=retrieval_k)

NameError: name 'types' is not defined

## 7) Save Artifacts for Web App

In [ ]:
PROJECT_ROOT = DATA_PATH.resolve().parents[2]
ARTIFACT_DIR = PROJECT_ROOT / "ml" / "artifacts" / "content_based_movie_v1"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# 1) Metadata table
metadata_cols = ["id", "title", "overview", "genres", "keywords", "top_5_actors", "director",
                 "collection_name", "vote_average", "vote_count", "poster_path",
                 "release_date", "runtime", "original_language", "has_franchise"]
metadata_cols = [c for c in metadata_cols if c in work_df.columns]

web_metadata = work_df[metadata_cols].copy()
web_metadata["id"] = pd.to_numeric(web_metadata["id"], errors="coerce").astype("int64")
web_metadata = web_metadata.reset_index().rename(columns={"index": "movie_index"})
web_metadata["genre_list"] = work_df["_genre_set"].apply(lambda v: sorted(list(v)))
web_metadata["keyword_list"] = work_df["_keyword_set"].apply(lambda v: sorted(list(v)))
web_metadata["actor_list"] = work_df["_actor_list"]

metadata_csv_path = ARTIFACT_DIR / "movie_metadata.csv"
metadata_parquet_path = ARTIFACT_DIR / "movie_metadata.parquet"
web_metadata.to_csv(metadata_csv_path, index=False)
web_metadata.to_parquet(metadata_parquet_path, index=False)

# 2) Embeddings
embeddings_path = ARTIFACT_DIR / "overview_embeddings.npy"
np.save(embeddings_path, embeddings.astype(np.float32))

# 3) Feature arrays
feature_arrays_path = ARTIFACT_DIR / "feature_arrays.npz"
np.savez(
    feature_arrays_path,
    bayesian_scores=bayesian_scores,
    bayesian_scores_norm=bayesian_scores_norm,
    recency_scores=recency_scores,
    ratings=ratings,
    votes=votes,
    movie_index=web_metadata["movie_index"].to_numpy(),
)

# 4) Config
config = {
    "artifact_version": "content_based_movie_v1",
    "model_name": MODEL_NAME,
    "created_at": pd.Timestamp.utcnow().isoformat(),
    "columns": {
        "title": "title",
        "overview": "overview",
        "genres": "genres",
        "keywords": "keywords",
        "actors": "top_5_actors",
        "director": "director",
        "collection": "collection_name",
        "vote_average": "vote_average",
        "vote_count": "vote_count",
        "release_date": "release_date",
        "poster": "poster_path",
    },
    "weights": BASE_FEATURE_WEIGHTS,
    "bayesian": {"C": C, "m": m, "normalized": True},
    "recency": {
        "decay_years": RECENCY_DECAY_YEARS,
        "usage": "booster_not_filter",
    },
    "retrieval": {
        "embedding_file": "overview_embeddings.npy",
        "similarity_runtime": "dot_product",
        "retrieval_k_default": RETRIEVAL_K_DEFAULT,
    },
    "gates": {
        "min_bayesian_score": MIN_BAYES_SCORE_GATE,
        "min_bayesian_with_strong_match": MIN_BAYES_WITH_STRONG_MATCH,
        "min_strong_metadata_match": MIN_STRONG_METADATA_GATE,
    },
    "rerank": {
        "quality_gate": "tightened_bayesian_gate_with_structured_penalties",
        "collection_caps": {"top10": 2, "top20": 3},
        "director_caps": {"top10": 2, "top20": 3},
        "title_family_caps": {"top10": 2, "top20": 3},
        "title_family_threshold": TITLE_FAMILY_MATCH_THRESHOLD,
        "penalties": {
            "collection_repeat": 0.12,
            "director_repeat": 0.05,
            "title_family_repeat": 0.12,
            "genre_redundancy": 0.08,
            "keyword_redundancy": 0.10,
            "embedding_redundancy": 0.08,
        },
    },
}

config_path = ARTIFACT_DIR / "config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Saved artifacts:")
for p in [metadata_csv_path, metadata_parquet_path, embeddings_path, feature_arrays_path, config_path]:
    print(f"  - {p}")

Metadata columns: ['anime_index', 'anime_id', 'Name', 'Synopsis', 'Genres', 'Type', 'Studios', 'Source', 'Score', 'Scored By', 'genre_list']
Metadata rows: 9185
Saved: /home/yigit-kayali/code/Projects/ML/Content Recommendation/ml/artifacts/content_based_anime_v1/anime_metadata.csv
Saved: /home/yigit-kayali/code/Projects/ML/Content Recommendation/ml/artifacts/content_based_anime_v1/anime_metadata.parquet


## 8) Demo (Adaptive Structured Recommendations)

In [ ]:
QUERY_NAME = "The Dark Knight"  # change this
TOP_K = 20
RETRIEVAL_K = 150

query_index = find_query_index(QUERY_NAME)
print("Query movie:", work_df.iloc[query_index]["title"])
print("Dynamic weights:", get_feature_weights(query_index))

demo_results = recommend_by_name(
    QUERY_NAME,
    top_k=TOP_K,
    retrieval_k=RETRIEVAL_K,
)
demo_results